# Самостоятельная работа: dense GAN на Fashion-MNIST

Здесь требуется самостоятельно перенести алгоритм состязательного
обучения на Fashion-MNIST. Данные отличаются от MNIST-примера, но
размер изображения, нормализация, архитектуры и математические цели
остаются теми же. Реализация завершена; все этапы и три запуска выполнены.

Выполняйте задания строго по порядку. На каждом этапе сначала
подтвердите форму тензоров и только потом продолжайте. Метки классов
Fashion-MNIST не становятся целями GAN: задача по-прежнему состоит в
различении реального и синтетического изображения.

## История одного пакета

Генератор превращает пачку случайных карточек `(B,100)` в пачку
картинок `(B,784)`. Дискриминатор смотрит и на эти картинки, и на
настоящие предметы одежды, а затем выдаёт по одному числу `(B,1)`.
Сначала учится проверяющий, потом — художник. Выполняйте этапы именно
по порядку: следующий этап использует объекты, созданные и проверенные
на предыдущем.

### Маленький словарь перед реализацией

Параметры — обучаемые числа в слоях. Прямой проход создаёт картинку и
ответ проверяющего. Потеря — одно число, описывающее ошибку текущего
ответа. Градиент — инструкция, как слегка поменять параметры, чтобы
ошибка уменьшилась. `GradientTape` находит такую инструкцию, а Adam
применяет небольшую поправку.

## Этап 1. Конфигурация независимого эксперимента

**Доступно:** константы ниже. **Цель:** зафиксировать Fashion-MNIST,
первые 10 000 изображений, $d_z=100$, $d_x=784$, пакет 128 и
первоначальные 105 эпох. Для сравнений позднее установите 205 и 400
эпох, не меняя остальную постановку. **Контрольная точка:** диапазон
целевых меток GAN — 0.9, 0 и 1; это не метки предметов одежды.

`(B,784)` означает «B картинок, каждая записана длинной строкой из
784 яркостей», а `(B,1)` — один ответ проверяющего на картинку. Это не
784 разных объектов и не десять классов Fashion-MNIST.

In [1]:
from __future__ import annotations

# Path указывает только место для сеток и графиков. NumPy/Matplotlib
# обслуживают данные и диагностику; TensorFlow строит граф и модели.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Зафиксируйте seed до первого шума и создания слоёв, иначе два
# запуска нельзя будет сравнивать по одним и тем же условиям.
SEED = 42
DATASET_NAME = 'Fashion-MNIST'
MAX_EXAMPLES = 10_000
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
# Размерность dense-входа: каждая карта 28×28 станет длиной 784.
IMAGE_DIM = IMAGE_HEIGHT * IMAGE_WIDTH
NOISE_DIM = 100
BATCH_SIZE = 128
EPOCHS = 105
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
FAKE_LABEL = 0.0
GENERATOR_TARGET = 1.0
SAVE_EVERY = 20

# Keras использует генератор случайных чисел и для весов, и для z.
tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)

## Этап 2. Получение и подготовка Fashion-MNIST

**Требуется:** получить обучающие изображения Fashion-MNIST, оставить
первые 10 000, преобразовать пиксели из $[0,255]$ в $[-1,1]$ и
развернуть их в $(N,784)$. Затем создайте перемешиваемый набор
мини-пакетов $(B,784)$ без меток классов. **Проверьте:** значения
конечны и принадлежат $[-1,1]$; метки используются только для
необязательной подписи исходных изображений.

**Маршрут:** сначала загрузите массив и посмотрите на одну картинку,
затем приведите значения к `float32`, измените шкалу `[0,255] → [-1,1]`
и лишь потом разверните карту. Останавливайтесь на `assert`: при
неверной форме или диапазоне сеть строить ещё рано. `y_train` не
передаётся GAN, иначе это была бы задача классификации одежды.

In [2]:
(raw_images, y_train), _ = tf.keras.datasets.fashion_mnist.load_data()
x_train = raw_images[:MAX_EXAMPLES].astype(np.float32)
y_train = y_train[:MAX_EXAMPLES]  # Только для диагностики; GAN метки не получает.
assert x_train.shape == (MAX_EXAMPLES, IMAGE_HEIGHT, IMAGE_WIDTH)
x_train = ((x_train - 127.5) / 127.5).reshape(MAX_EXAMPLES, IMAGE_DIM)
dataset = (tf.data.Dataset.from_tensor_slices(x_train)
           .shuffle(MAX_EXAMPLES, seed=SEED, reshuffle_each_iteration=True)
           .batch(BATCH_SIZE, drop_remainder=False)
           .prefetch(tf.data.AUTOTUNE))
first_batch = next(iter(dataset))
assert first_batch.shape == (BATCH_SIZE, IMAGE_DIM)
assert np.isfinite(x_train).all() and -1 <= x_train.min() <= x_train.max() <= 1
print('Данные:', x_train.shape, 'пакет:', first_batch.shape,
      'диапазон:', (x_train.min(), x_train.max()))

Данные: (10000, 784) пакет: (128, 784) диапазон: (np.float32(-1.0), np.float32(1.0))


## Этап 3. Генератор и проверка его выхода

**Требуется:** собрать генератор
$100\to256\to512\to1024\to784$ с тремя LeakyReLU и финальным tanh.
Вход — шум $(B,100)$ с независимыми нормальными координатами; выход —
синтетические развёрнутые изображения $(B,784)$ в $[-1,1]$.
Используйте современный tf.keras и не применяйте готовый GAN-слой.
**Контрольная точка:** выход имеет точную форму и не содержит NaN или
бесконечностей.

Считайте `z` карточкой со 100 случайными ручками. В начале генератор
рисует бессмыслицу; обучение понемногу поправляет его веса, чтобы те
же 100 чисел превращались в структуру, похожую на данные. `tanh` не
даёт пикселям покинуть диапазон настоящих картинок.

In [3]:
def make_generator():
    layers = [tf.keras.Input(shape=(NOISE_DIM,))]
    for width in (256, 512, 1024):
        layers += [tf.keras.layers.Dense(width, kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)),
                   tf.keras.layers.LeakyReLU(negative_slope=0.2)]
    layers.append(tf.keras.layers.Dense(IMAGE_DIM, activation='tanh',
                  kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)))
    return tf.keras.Sequential(layers, name='fashion_generator')

generator = make_generator()
z_check = tf.random.normal((BATCH_SIZE, NOISE_DIM))
fake_check = generator(z_check, training=False)
assert fake_check.shape == (BATCH_SIZE, IMAGE_DIM)
assert bool(tf.reduce_all(tf.math.is_finite(fake_check)))
assert bool(tf.reduce_all(tf.abs(fake_check) <= 1))
print('G:', z_check.shape, '→', fake_check.shape)

G: (128, 100) → (128, 784)


## Этап 4. Дискриминатор и смысл его выхода

**Требуется:** собрать дискриминатор
$784\to1024\to512\to256\to1$ с LeakyReLU, Dropout(0.3) и sigmoid.
Его входом по очереди будут настоящие и синтетические изображения
$(B,784)$, а результатом — $(B,1)$ вероятностей. Последняя координата
— не класс Fashion-MNIST, а оценка «настоящее».
**Контрольная точка:** на диагностическом входе значения строго между
0 и 1.

Единственный вопрос дискриминатора — «эта картинка пришла из набора
или создана генератором?». Его ответ после sigmoid можно читать как
уверенность от 0 до 1. Это совсем не вероятность одного из классов
одежды и не оценка того, насколько красив предмет.

In [4]:
def make_discriminator():
    layers = [tf.keras.Input(shape=(IMAGE_DIM,))]
    for width in (1024, 512, 256):
        layers += [tf.keras.layers.Dense(width, kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)),
                   tf.keras.layers.LeakyReLU(negative_slope=0.2),
                   tf.keras.layers.Dropout(0.3)]
    layers.append(tf.keras.layers.Dense(1, activation='sigmoid',
                  kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02)))
    return tf.keras.Sequential(layers, name='fashion_discriminator')

discriminator = make_discriminator()
for images in (first_batch, fake_check):
    prediction = discriminator(images, training=False)
    assert prediction.shape == (BATCH_SIZE, 1)
    assert bool(tf.reduce_all(tf.math.is_finite(prediction)))
    assert bool(tf.reduce_all((prediction > 0) & (prediction < 1)))
print('D:', prediction.shape, 'вероятности:',
      (float(tf.reduce_min(prediction)), float(tf.reduce_max(prediction))))

D: (128, 1) вероятности: (0.49989187717437744, 0.5000060796737671)


## Этап 5. Две функции потерь и два независимых обновления

**Требуется:** создать BinaryCrossentropy с «from_logits=False» и два
Adam-оптимизатора с заданными learning_rate и beta_1. Напишите
попеременный шаг: сначала обновляет только дискриминатор по настоящим
меткам 0.9 и синтетическим 0; затем обновляет только генератор по
цели 1. Для первого шага отделите синтетические изображения от
градиента генератора. Для второго вызовите дискриминатор с
«training=False», но сохраните прохождение градиента к генератору.

**Контрольная точка:** после первого подшага меняются только
переменные дискриминатора, после второго — только переменные
генератора. Обе потери должны быть конечными скалярами.

**Сначала → затем → проверить.** Сначала выполните фазу D и отделите
созданные картинки через `stop_gradient`: обратная стрелка к G
выключена. Затем создайте новый `GradientTape` для фазы G: стрелка от
ошибки идёт через D к G, но список параметров оптимизатора содержит
только G. Визуальная проверка проста: спросите, какие веса стоят справа
в каждом вызове `apply_gradients`.

Здесь D возвращает вероятность $p$ после sigmoid. Для одной цели $y$
API `BinaryCrossentropy(from_logits=False)` использует

$$
\operatorname{BCE}(y,p)=-y\log p-(1-y)\log(1-p).
$$

Неправильный уверенный ответ получает большую потерю; правильный
уверенный ответ — малую. Для всего мини-пакета TensorFlow усредняет
эту ошибку, получая один скаляр для каждого игрока.

### Математический контракт вашей реализации

**Сначала посмотрите на две фазы глазами данных.**

![Два независимых обновления GAN: обучение дискриминатора, затем обучение генератора](assets/diagrams/gan_two_phase_training.svg)

Эта схема разделяет временные тензоры, скалярные потери и параметры.
Ваша будущая функция не должна пытаться «обучать шум» или метки
Fashion-MNIST: шум — вход, а метки классов вообще не участвуют в GAN.
Обучаемыми остаются только `tf.Variable` внутри G и D.

Обозначим веса и смещения генератора через $\theta_G$, а параметры
дискриминатора — через $\theta_D$. Потеря — один скаляр. Производная
по параметру показывает направление его маленькой поправки:

$$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L.
$$

В первой фазе для мини-пакета должны быть вычислены два члена
$\operatorname{BCE}(0.9,D(\mathbf x_{\rm real}))$ и
$\operatorname{BCE}(0,D(G(\mathbf z_D)))$; их среднее образует
$\mathcal L_D$, а градиент запрашивается только по $\theta_D$.
Во второй фазе $\mathcal L_G=\operatorname{BCE}(1,D(G(\mathbf z_G)))$
дифференцируется только по $\theta_G$. Шум — не параметр: каждый
$\mathbf z$ является новым входом формы `(B,100)`.

### Как `GradientTape` превращает loss в поправки весов

![Жизненный цикл GradientTape от прямого прохода до обновления Adam](assets/diagrams/tensorflow_gradient_tape.svg)

`GradientTape` записывает прямой путь вычислений до `loss`, а не
делает отдельное обучение. Пока активен блок `with`, он помнит
дифференцируемые операции и автоматически наблюдает используемые
обучаемые `tf.Variable`. Затем `tape.gradient(loss, variables)`
запускает обратный проход и возвращает по одной производной для
каждого элемента `variables`; форма градиента совпадает с формой
веса. `zip` сопоставляет эти два списка, а `apply_gradients` передаёт
пары Adam. Именно `variables` определяет, обновляется $\theta_G$ или
$\theta_D$; имя tape этого решения не принимает.

Непостоянный tape имеет одно назначение: он нужен для одного вызова
`gradient`, затем его запись освобождается. Это программная запись
правила градиентного шага, а не эвристика TensorFlow. Производные
получаются по цепному правилу: библиотека не выписывает гигантские
якобианы, а передаёт их нужное действие от скалярной потери к весам.

### Один мини-пакет: карта объектов вашей реализации

Используйте эти имена как ориентиры при заполнении TODO. Объекты с
пометкой «временный» не являются параметрами и не передаются Adam.

| Математика | Ожидаемое имя | Форма | Роль в одной фазе |
| --- | --- | --- | --- |
| $\mathbf x_{\rm real}$ | `real_images` | `(B,784)`, `float32` | Настоящие картинки для D. |
| $\mathbf z_D$ | `discriminator_noise` | `(B,100)` | Временный шум фазы D. |
| $G(\mathbf z_D)$ | `fake_images` | `(B,784)` | Подделки после `stop_gradient`. |
| $D(\mathbf x)$ | `real_scores`, `fake_scores` | `(B,1)`, вероятность | Аргумент BCE фазы D. |
| $\mathcal L_D$ | `discriminator_loss` | скаляр | Даёт градиенты только для `discriminator.trainable_variables`. |
| $\mathbf z_G$ | `generator_noise` | `(B,100)` | Новый шум, независимый от $\mathbf z_D$. |
| $G(\mathbf z_G)$ | `generated_images` | `(B,784)` | Подделки, оставленные в графе. |
| $\mathcal L_G$ | `generator_loss` | скаляр | Даёт градиенты только для `generator.trainable_variables`. |

**Проведите одну фазу буквально по строкам.** Сначала внутри первого
`GradientTape` вычисляются входы D и один скаляр `discriminator_loss`.
Затем `tape.gradient(discriminator_loss,
discriminator.trainable_variables)` строит список производных в том же
порядке, что список весов. `zip` образует пары, и только после
`discriminator_optimizer.apply_gradients(...)` меняются $\theta_D$.
Фаза G повторяет эту схему с новым tape, новым шумом и списком весов G.

Для этой модели D выдаёт вероятность, поэтому точный путь градиента
генератора можно записать через якобианы:

$$
\nabla_{\theta_G}\mathcal L_G
=J_{G,\theta_G}^{\mathsf T}J_{D,\mathbf x}^{\mathsf T}
\nabla_{\mathbf p}\mathcal L_G.
$$

Якобиан — таблица всех локальных влияний: как выход меняется при малом
изменении входа. Он слишком велик для явного создания; `gradient`
вычисляет только нужный обратный проход, то есть произведения с ним.

In [5]:
def make_optimizers():
    return (tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1),
            tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1))

bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
generator_optimizer, discriminator_optimizer = make_optimizers()

def make_train_step(generator, discriminator, generator_optimizer, discriminator_optimizer):
    # Отдельный tf.function для каждого независимого запуска.
    @tf.function
    def train_step(real_images):
        batch_size = tf.shape(real_images)[0]
        z_d = tf.random.normal((batch_size, NOISE_DIM))
        with tf.GradientTape() as tape_d:
            fake_images = tf.stop_gradient(generator(z_d, training=False))
            real_scores = discriminator(real_images, training=True)
            fake_scores = discriminator(fake_images, training=True)
            loss_d = 0.5 * (bce(tf.ones_like(real_scores) * REAL_LABEL, real_scores)
                            + bce(tf.zeros_like(fake_scores), fake_scores))
        grads_d = tape_d.gradient(loss_d, discriminator.trainable_variables)
        discriminator_optimizer.apply_gradients(zip(grads_d, discriminator.trainable_variables))

        z_g = tf.random.normal((batch_size, NOISE_DIM))
        with tf.GradientTape() as tape_g:
            generated = generator(z_g, training=True)
            scores = discriminator(generated, training=False)
            loss_g = bce(tf.ones_like(scores) * GENERATOR_TARGET, scores)
        grads_g = tape_g.gradient(loss_g, generator.trainable_variables)
        generator_optimizer.apply_gradients(zip(grads_g, generator.trainable_variables))
        return loss_d, loss_g
    return train_step

assert set(map(id, generator.trainable_variables)).isdisjoint(map(id, discriminator.trainable_variables))
print('Оптимизаторы разделены; фаза D обновляет D, фаза G обновляет G.')

Оптимизаторы разделены; фаза D обновляет D, фаза G обновляет G.


## Этап 6. Эксперименты, сетки и отчёт

**Требуется:** обучите GAN сначала 105 эпох. На фиксированных 100
шумовых векторах сохраните сетки после первой эпохи, далее через
равные интервалы и в конце. Постройте отдельные ряды потерь
дискриминатора и генератора. Затем повторите запуск для 205 и 400 эпох
с теми же архитектурой, шумом, размером выборки и способом
визуализации.

**Проверка:** все значения потерь конечны; формы не меняются;
изображения становятся разнообразнее или обнаруживаются и
объясняются признаки неустойчивости. В отчёте сравните три режима, не
объявляя меньшую потерю одной стороны безусловной победой GAN.

Фиксированный шум — одинаковый набор заданий для художника. Показывая
его после нескольких эпох, вы видите изменение весов, а не новую
случайную карточку. Для Matplotlib значения возвращаются из `[-1,1]`
в `[0,1]`: модель при этом не меняется. Кривые потерь читайте вместе с
сетками, потому что GAN — соревнование, а не одна убывающая ошибка.

In [6]:
import csv

def save_grid(model, noise, path):
    images = (model(noise, training=False).numpy().reshape(10, 10, 28, 28) + 1) / 2
    fig, axes = plt.subplots(10, 10, figsize=(10, 10))
    for ax, image in zip(axes.flat, images.reshape(100, 28, 28)):
        ax.imshow(image, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
    fig.subplots_adjust(wspace=0.02, hspace=0.02)
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)

def run_experiment(epochs, run_dir, fixed_noise):
    global generator, discriminator, generator_optimizer, discriminator_optimizer
    tf.keras.utils.set_random_seed(SEED)  # Одинаковые начальные веса для честного сравнения.
    generator, discriminator = make_generator(), make_discriminator()
    generator_optimizer, discriminator_optimizer = make_optimizers()
    step = make_train_step(generator, discriminator, generator_optimizer, discriminator_optimizer)
    run_dir.mkdir(parents=True, exist_ok=True)
    history = []
    for epoch in range(1, epochs + 1):
        losses = [tuple(float(x.numpy()) for x in step(batch)) for batch in dataset]
        mean_d, mean_g = np.mean(losses, axis=0)
        assert np.isfinite((mean_d, mean_g)).all()
        history.append((epoch, mean_d, mean_g))
        if epoch == 1 or epoch % SAVE_EVERY == 0 or epoch == epochs:
            save_grid(generator, fixed_noise, run_dir / f'generated_epoch_{epoch:03d}.png')
            print(f'{epochs} эпох: {epoch:3d}, D={mean_d:.4f}, G={mean_g:.4f}')
    with (run_dir / 'losses.csv').open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(('epoch', 'discriminator_loss', 'generator_loss'))
        writer.writerows(history)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot([r[0] for r in history], [r[1] for r in history], label='D')
    ax.plot([r[0] for r in history], [r[2] for r in history], label='G')
    ax.set(xlabel='Эпоха', ylabel='BCE', title=f'Fashion-MNIST, {epochs} эпох')
    ax.legend(); ax.grid(alpha=0.3)
    fig.savefig(run_dir / 'losses.png', dpi=160, bbox_inches='tight')
    plt.close(fig)
    return history

tf.keras.utils.set_random_seed(SEED + 1)
fixed_noise = tf.random.normal((100, NOISE_DIM))
histories = {}
for epochs in (105, 205, 400):
    histories[epochs] = run_experiment(epochs, artifact_dir / f'epochs_{epochs}', fixed_noise)
print('Финальные потери:', {n: h[-1][1:] for n, h in histories.items()})

105 эпох:   1, D=0.5286, G=0.7788


105 эпох:  20, D=0.4866, G=2.1075


105 эпох:  40, D=0.5277, G=1.6863


105 эпох:  60, D=0.5801, G=1.3827


105 эпох:  80, D=0.6034, G=1.2565


105 эпох: 100, D=0.6152, G=1.2077


... (30 outputs left out)

400 эпох: 340, D=0.6267, G=1.1234


400 эпох: 360, D=0.6234, G=1.1479


400 эпох: 380, D=0.6277, G=1.1470


400 эпох: 400, D=0.6239, G=1.1708
Финальные потери: {105: (np.float64(0.6230491686470901), np.float64(1.1748809814453125)), 205: (np.float64(0.635557009449488), np.float64(1.082782342464109)), 400: (np.float64(0.6238622280615794), np.float64(1.1707984616484823))}


## Этап 7. Итоговая самопроверка

- Использован именно Fashion-MNIST, а не MNIST из примера.
- Шум имеет форму $(B,100)$; реальные и синтетические изображения —
  $(B,784)$; ответ дискриминатора — $(B,1)$.
- Реальные значения нормированы к $[-1,1]$, как и выход tanh
  генератора.
- Метки классов одежды не входят в BinaryCrossentropy GAN.
- Генератор и дискриминатор обновляются отдельными оптимизаторами.
- Сетки и две истории потерь сохранены для всех трёх экспериментов.

## Итог выполненных запусков

Все запуски используют Fashion-MNIST и фиксированный шум для сеток. Финальные потери D/G: 105 эпох — 0,623/1,175; 205 — 0,636/1,083; 400 — 0,624/1,171. Сетки сохраняют разные силуэты, но дополнительное обучение не даёт явного улучшения.

- [Сетка 105 эпох](artifacts/fashion_mnist_dense_gan/epochs_105/generated_epoch_105.png) и [график потерь](artifacts/fashion_mnist_dense_gan/epochs_105/losses.png)
- [Сетка 205 эпох](artifacts/fashion_mnist_dense_gan/epochs_205/generated_epoch_205.png) и [график потерь](artifacts/fashion_mnist_dense_gan/epochs_205/losses.png)
- [Сетка 400 эпох](artifacts/fashion_mnist_dense_gan/epochs_400/generated_epoch_400.png) и [график потерь](artifacts/fashion_mnist_dense_gan/epochs_400/losses.png)

Полные значения потерь находятся в `losses.csv` каждого каталога.